<a href="https://colab.research.google.com/github/Dhay-1/GP2/blob/Dhay-Branch/Copy_of_AmanPlayM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers arabert farasapy emoji==1.4.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 20.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 9.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from arabert.preprocess import ArabertPreprocessor

# Hardware Conformation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The device is: {device}")

The device is: cuda


In [ ]:
print(os.listdir('/content/drive/MyDrive/Colab_Notebooks'))

['Copy of AmanPlayM.ipynb', 'train_Arabic_tweets_positive_20190413.tsv', 'CS351T- project phase 2.docx', 'HCIproject.pdf', 'Pre_AI_Lab1 (1).ipynb', 'Copy of Welcome To Colab', 'Untitled0.ipynb', 'Pre_AI_Lab1.ipynb', 'AmanPlay_GP2.ipynb', 'AmanPlay_Positive_Only.xlsx', 'Final_Data_Segmented.xlsx']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_bullying = pd.read_excel('/content/drive/MyDrive/Colab_Notebooks/Final_Data_Segmented.xlsx')
df_bullying = pd.DataFrame({'Text': df_bullying['Comments'], 'Label': 1})

df_kind = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/train_Arabic_tweets_positive_20190413.tsv', sep='\t', header=None, names=['Label_Name', 'Text'])
df_kind = pd.DataFrame({'Text': df_kind['Text'], 'Label': 0}).head(len(df_bullying))

df = pd.concat([df_bullying, df_kind], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)

In [ ]:
model_name = "aubmindlab/bert-base-arabertv2"
prep_tool = ArabertPreprocessor(model_name=model_name)

df['Text'] = df['Text'].apply(lambda x: prep_tool.preprocess(str(x)))

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'farasa-api.qcri.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


100%|██████████| 241M/241M [05:28<00:00, 735kiB/s]


[2026-02-08 17:10:10,292 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Text'].values,
    df['Label'].values,
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_data(texts):

    list_texts = [str(t) for t in texts]

    return tokenizer(
        list_texts,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

# Encoding
train_encodings = tokenize_data(train_texts)
test_encodings = tokenize_data(test_texts)

train_labels_tensor = torch.tensor(train_labels)
test_labels_tensor = torch.tensor(test_labels)

print("Done!!!!.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Done!!!!.


In [ ]:
import torch.nn.functional as F

class HybridModel(nn.Module):
    def __init__(self, model_name):
        super(HybridModel, self).__init__()
        # 1.AraBert Layer
        self.bert = AutoModel.from_pretrained(model_name)

        # 2.CNN layer
        self.cnn = nn.Conv1d(768, 128, kernel_size=3, padding=1)

        # 3.Bi-LSTM layer
        self.lstm = nn.LSTM(768, 128, bidirectional=True, batch_first=True)

        # 4.GRU Layer
        self.gru = nn.GRU(768, 128, batch_first=True)

        # 5. the result after collected
        #output: BERT(768) + CNN(128) + LSTM(256) + GRU(128) = 1280
        self.classifier = nn.Linear(768 + 128 + 256 + 128, 2)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        pooled_output = outputs.pooler_output

        #  CNN proccesing
        cnn_in = sequence_output.permute(0, 2, 1)
        cnn_out = F.relu(self.cnn(cnn_in)).max(dim=2)[0]

        #  Bi-LSTM proccessing
        lstm_out, _ = self.lstm(sequence_output)
        lstm_out = lstm_out[:, -1, :]

        #  GRU proccesing
        gru_out, _ = self.gru(sequence_output)
        gru_out = gru_out[:, -1, :]

        # Integraion
        combined = torch.cat((pooled_output, cnn_out, lstm_out, gru_out), dim=1)

        #Bullying ?
        return self.classifier(combined)

#(GPU)
model = HybridModel(model_name).to(device)
print("The HybirdModel is Ready!")

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The HybirdModel is Ready!


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim


train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], train_labels_tensor)
test_dataset = TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)


optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print("The trainning will be ready!")

The trainning will be ready!


In [ ]:
# Training parameters
epochs = 3

print("Starting the Hybrid AraBERT Training Process...")
print("-" * 40)

for epoch in range(epochs):
    model.train()
    total_loss = 0

    # Iterate through the Arabic text batches
    for batch in train_loader:
        optimizer.zero_grad()

        # Move Arabic data tensors to GPU
        input_ids = batch[0].to(device)
        mask = batch[1].to(device)
        labels = batch[2].to(device)

        # Model processes the Arabic text here
        outputs = model(input_ids, mask)
        loss = criterion(outputs, labels)

        # Update model weights based on Arabic language patterns
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Calculate average loss for the current epoch
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Average Loss: {avg_loss:.4f}")

print("-" * 40)
print("Training Completed Successfully!")

Starting the Hybrid AraBERT Training Process...
----------------------------------------
Epoch 1/3 | Average Loss: 0.2323
Epoch 2/3 | Average Loss: 0.0980
Epoch 3/3 | Average Loss: 0.0483
----------------------------------------
Training Completed Successfully!


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

model.eval() # Set the model to evaluation mode
all_preds = []
all_labels = []

print("Starting Evaluation on Test Data...")

with torch.no_grad(): # Disable gradient calculation for faster inference
    for batch in test_loader:
        input_ids = batch[0].to(device)
        mask = batch[1].to(device)
        labels = batch[2].to(device)

        # Get predictions
        outputs = model(input_ids, mask)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate Metrics
accuracy = accuracy_score(all_labels, all_preds)
print("-" * 30)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print("-" * 30)
print("Detailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['NonBullying', 'Bullying']))

Starting Evaluation on Test Data...
------------------------------
Test Accuracy: 95.89%
------------------------------
Detailed Classification Report:
              precision    recall  f1-score   support

 NonBullying       0.95      0.97      0.96       572
    Bullying       0.97      0.94      0.96       546

    accuracy                           0.96      1118
   macro avg       0.96      0.96      0.96      1118
weighted avg       0.96      0.96      0.96      1118



In [ ]:
def predict_bullying(text):
    model.eval()
    # 1. Preprocess & Tokenize the input text
    inputs = tokenizer(prep_tool.preprocess(text), return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    # 2. Forward pass to get predictions
    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])
        prediction = torch.argmax(outputs, dim=1).item()

    # 3. Print the result in English logs
    result = "Bullying" if prediction == 1 else "non Bullying"
    print(f"Input Text: {text}")
    print(f"Detection Result: {result}")
    print("-" * 30)

# --- Try some Arabic phrases here ---
predict_bullying("غبي")
predict_bullying("انت شخص ذكي")

Input Text: غبي
Detection Result: Bullying
------------------------------
Input Text: انت شخص ذكي
Detection Result: non Bullying
------------------------------


In [ ]:
def predict_bullying_with_score(text):
    model.eval()
    inputs = tokenizer(prep_tool.preprocess(text), return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])

        probs = F.softmax(outputs, dim=1)
        confidence, prediction = torch.max(probs, dim=1)

    result = "Bullying" if prediction.item() == 1 else "Non-Bullying"
    print(f"Text: {text}")
    print(f"Result: {result} (Confidence: {confidence.item()*100:.2f}%)")
    print("-" * 30)


predict_bullying_with_score("غبي")

Text: غبي
Result: Bullying (Confidence: 99.84%)
------------------------------


In [ ]:
import torch
import os


save_path = "AmanPlay_Final_Model"
if not os.path.exists(save_path):
    os.makedirs(save_path)

# (Weights)

torch.save(model.state_dict(), os.path.join(save_path, "AmanPlay_weights.pt"))


tokenizer.save_pretrained(save_path)

print(f"Done! AmanPlay is saved in: {save_path}")

Done! AmanPlay is saved in: AmanPlay_Final_Model


In [ ]:
import shutil
from google.colab import files


folder_name = "AmanPlay_Final_Model"


shutil.make_archive('AmanPlay_Desktop_Backup', 'zip', folder_name)


files.download('AmanPlay_Desktop_Backup.zip')

print(" ...  in Downloads  ")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 ...  in Downloads  


In [ ]:
# Reem: ASR (whisper model)

!pip install scipy librosa noisereduce torch transformers soundfile
!pip install openai-whisper

from google.colab import files
uploaded = files.upload()

import librosa
import noisereduce as nr
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import soundfile as sf


# upLoad audio

audio_path = list(uploaded.keys())[0]

speech, sr = librosa.load(audio_path, sr=16000)



# Noise cleaning

noise = speech[:int(0.5 * sr)]

clean_speech = nr.reduce_noise(y=speech, y_noise=noise, sr=sr)

sf.write('clean_audio.wav', clean_speech, sr)

print("clean_audio.wav created")


# Load ASR model using whisper

import whisper
model = whisper.load_model("base") # small / medium also possible
result = model.transcribe( "clean_audio.wav", language="ar" )
transcription = result["text"]
print("Transcription:")
print(transcription)




with open("transcription.txt", "w", encoding="utf-8") as f:

    f.write(transcription)

print("Saved transcription.txt")





Saving Recording.m4a to Recording (5).m4a
clean_audio.wav created


/tmp/ipython-input-3475948245.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, sr = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


📝 Transcription:
 سلام عليكم و نسنيني
✔ Saved transcription.txt


In [ ]:
# Wajd Dense layer

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

model = Sequential()
model.add(Flatten(input_shape=(28,28)))
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(10, activation='softmax')) #output layer for ten classes

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)